In [17]:
"""
ETL Pipeline - PokéAPI
Arquitectura Medallion: Bronze → Silver → Gold
"""

import requests
import pandas as pd
import json
from datetime import datetime
from typing import Dict, List
import time
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from typing import Dict
import os

In [2]:


# ============================================
# CAPA BRONZE - Extracción de datos raw
# ============================================

class BronzeLayer:
    """Extrae datos crudos de la API y los almacena sin transformar"""

    def __init__(self):
        self.base_url = "https://pokeapi.co/api/v2"
        self.bronze_data = {}

    def extract_pokemon_data(self, limit: int = 50) -> Dict:
        """Extrae información básica de pokémon"""
        print(f"🔹 BRONZE: Extrayendo datos de {limit} pokémon...")

        try:
            # Obtener lista de pokémon
            response = requests.get(f"{self.base_url}/pokemon?limit={limit}")
            response.raise_for_status()
            pokemon_list = response.json()['results']

            # Extraer detalles de cada pokémon
            pokemon_details = []
            for i, pokemon in enumerate(pokemon_list, 1):
                print(f"  Extrayendo {i}/{limit}: {pokemon['name']}", end='\r')
                detail_response = requests.get(pokemon['url'])
                if detail_response.status_code == 200:
                    pokemon_details.append(detail_response.json())
                time.sleep(0.1)  # Rate limiting

            print("\n✅ BRONZE: Extracción completada")

            self.bronze_data = {
                'extraction_timestamp': datetime.now().isoformat(),
                'source': 'pokeapi.co',
                'pokemon_count': len(pokemon_details),
                'raw_data': pokemon_details
            }

            return self.bronze_data

        except requests.exceptions.RequestException as e:
            print(f"❌ Error en extracción: {e}")
            return {}

    def save_bronze(self, filename: str = 'pokemon_bronze.json'):
        """Guarda datos bronze en formato JSON"""
        with open(filename, 'w') as f:
            json.dump(self.bronze_data, f, indent=2)
        print(f"💾 BRONZE: Datos guardados en {filename}")


In [3]:
# ============================================
# CAPA SILVER - Limpieza y normalización
# ============================================

class SilverLayer:
    """Transforma y limpia los datos del bronze layer"""

    def __init__(self, bronze_data: Dict):
        self.bronze_data = bronze_data
        self.silver_dfs = {}

    def transform_pokemon_base(self) -> pd.DataFrame:
        """Transforma datos base de pokémon"""
        print("\n🔸 SILVER: Transformando datos base...")

        raw_data = self.bronze_data.get('raw_data', [])

        pokemon_base = []
        for pokemon in raw_data:
            pokemon_base.append({
                'pokemon_id': pokemon['id'],
                'name': pokemon['name'].title(),
                'height': pokemon['height'] / 10,  # Convertir a metros
                'weight': pokemon['weight'] / 10,  # Convertir a kg
                'base_experience': pokemon['base_experience'],
                'is_default': pokemon['is_default']
            })

        df = pd.DataFrame(pokemon_base)
        print(f"✅ SILVER: {len(df)} pokémon procesados")
        return df

    def transform_pokemon_types(self) -> pd.DataFrame:
        """Extrae y normaliza tipos de pokémon"""
        print("🔸 SILVER: Transformando tipos...")

        raw_data = self.bronze_data.get('raw_data', [])

        types_data = []
        for pokemon in raw_data:
            for type_info in pokemon['types']:
                types_data.append({
                    'pokemon_id': pokemon['id'],
                    'pokemon_name': pokemon['name'].title(),
                    'type': type_info['type']['name'].title(),
                    'slot': type_info['slot']
                })

        df = pd.DataFrame(types_data)
        print(f"✅ SILVER: {len(df)} relaciones de tipo procesadas")
        return df

    def transform_pokemon_stats(self) -> pd.DataFrame:
        """Extrae y normaliza estadísticas"""
        print("🔸 SILVER: Transformando estadísticas...")

        raw_data = self.bronze_data.get('raw_data', [])

        stats_data = []
        for pokemon in raw_data:
            for stat in pokemon['stats']:
                stats_data.append({
                    'pokemon_id': pokemon['id'],
                    'pokemon_name': pokemon['name'].title(),
                    'stat_name': stat['stat']['name'].replace('-', ' ').title(),
                    'base_stat': stat['base_stat'],
                    'effort': stat['effort']
                })

        df = pd.DataFrame(stats_data)
        print(f"✅ SILVER: {len(df)} estadísticas procesadas")
        return df

    def transform_pokemon_abilities(self) -> pd.DataFrame:
        """Extrae y normaliza habilidades"""
        print("🔸 SILVER: Transformando habilidades...")

        raw_data = self.bronze_data.get('raw_data', [])

        abilities_data = []
        for pokemon in raw_data:
            for ability in pokemon['abilities']:
                abilities_data.append({
                    'pokemon_id': pokemon['id'],
                    'pokemon_name': pokemon['name'].title(),
                    'ability': ability['ability']['name'].replace('-', ' ').title(),
                    'is_hidden': ability['is_hidden'],
                    'slot': ability['slot']
                })

        df = pd.DataFrame(abilities_data)
        print(f"✅ SILVER: {len(df)} habilidades procesadas")
        return df

    def process_all(self) -> Dict[str, pd.DataFrame]:
        """Procesa todas las transformaciones"""
        self.silver_dfs = {
            'pokemon_base': self.transform_pokemon_base(),
            'pokemon_types': self.transform_pokemon_types(),
            'pokemon_stats': self.transform_pokemon_stats(),
            'pokemon_abilities': self.transform_pokemon_abilities()
        }
        return self.silver_dfs

    def save_silver(self, prefix: str = 'pokemon_silver'):
        """Guarda los dataframes silver como CSV"""
        for name, df in self.silver_dfs.items():
            filename = f"{prefix}_{name}.csv"
            df.to_csv(filename, index=False)
            print(f"💾 SILVER: {name} guardado en {filename}")

In [4]:
# ============================================
# CAPA GOLD - Agregaciones y análisis
# ============================================

class GoldLayer:
    """Crea datasets optimizados para análisis"""

    def __init__(self, silver_dfs: Dict[str, pd.DataFrame]):
        self.silver_dfs = silver_dfs
        self.gold_dfs = {}

    def create_pokemon_summary(self) -> pd.DataFrame:
        """Crea resumen completo de pokémon con métricas agregadas"""
        print("\n🔶 GOLD: Creando resumen de pokémon...")

        base = self.silver_dfs['pokemon_base'].copy()

        # Agregar tipos (concatenar múltiples tipos)
        types_pivot = self.silver_dfs['pokemon_types'].groupby('pokemon_id')['type'].apply(
            lambda x: ' / '.join(sorted(x))
        ).reset_index()
        types_pivot.columns = ['pokemon_id', 'types']

        # Agregar estadísticas (calcular total y promedio)
        stats = self.silver_dfs['pokemon_stats'].copy()
        stats_agg = stats.groupby('pokemon_id').agg({
            'base_stat': ['sum', 'mean', 'max']
        }).reset_index()
        stats_agg.columns = ['pokemon_id', 'total_stats', 'avg_stat', 'max_stat']

        # Contar habilidades
        abilities_count = self.silver_dfs['pokemon_abilities'].groupby('pokemon_id').size().reset_index()
        abilities_count.columns = ['pokemon_id', 'abilities_count']

        # Merge todo
        summary = base.merge(types_pivot, on='pokemon_id', how='left')
        summary = summary.merge(stats_agg, on='pokemon_id', how='left')
        summary = summary.merge(abilities_count, on='pokemon_id', how='left')

        # Agregar categorías
        summary['size_category'] = pd.cut(
            summary['height'],
            bins=[0, 1, 2, float('inf')],
            labels=['Small', 'Medium', 'Large']
        )

        summary['power_level'] = pd.cut(
            summary['total_stats'],
            bins=[0, 300, 400, 500, float('inf')],
            labels=['Low', 'Medium', 'High', 'Elite']
        )

        print(f"✅ GOLD: Resumen creado con {len(summary)} pokémon")
        return summary

    def create_type_analysis(self) -> pd.DataFrame:
        """Análisis por tipo de pokémon"""
        print("🔶 GOLD: Creando análisis por tipo...")

        types = self.silver_dfs['pokemon_types'].copy()
        base = self.silver_dfs['pokemon_base'].copy()

        # Merge con datos base
        type_data = types.merge(base, on='pokemon_id', how='left')

        # Análisis agregado por tipo
        type_analysis = type_data.groupby('type').agg({
            'pokemon_id': 'count',
            'height': 'mean',
            'weight': 'mean',
            'base_experience': 'mean'
        }).reset_index()

        type_analysis.columns = [
            'type', 'pokemon_count', 'avg_height_m', 'avg_weight_kg', 'avg_experience'
        ]

        type_analysis = type_analysis.sort_values('pokemon_count', ascending=False)

        print(f"✅ GOLD: Análisis de {len(type_analysis)} tipos creado")
        return type_analysis

    def create_stats_pivot(self) -> pd.DataFrame:
        """Tabla pivote de estadísticas por pokémon"""
        print("🔶 GOLD: Creando pivot de estadísticas...")

        stats = self.silver_dfs['pokemon_stats'].copy()

        stats_pivot = stats.pivot_table(
            index=['pokemon_id', 'pokemon_name'],
            columns='stat_name',
            values='base_stat',
            aggfunc='first'
        ).reset_index()

        # Calcular total
        stat_cols = [col for col in stats_pivot.columns if col not in ['pokemon_id', 'pokemon_name']]
        stats_pivot['Total'] = stats_pivot[stat_cols].sum(axis=1)

        print(f"✅ GOLD: Pivot creado con {len(stats_pivot)} pokémon")
        return stats_pivot

    def process_all(self) -> Dict[str, pd.DataFrame]:
        """Procesa todas las agregaciones"""
        self.gold_dfs = {
            'pokemon_summary': self.create_pokemon_summary(),
            'type_analysis': self.create_type_analysis(),
            'stats_pivot': self.create_stats_pivot()
        }
        return self.gold_dfs

    def save_gold(self, prefix: str = 'pokemon_gold'):
        """Guarda los dataframes gold como CSV"""
        for name, df in self.gold_dfs.items():
            filename = f"{prefix}_{name}.csv"
            df.to_csv(filename, index=False)
            print(f"💾 GOLD: {name} guardado en {filename}")

In [19]:
# ============================================
# CAPA VISUALIZATION - Dashboards y Gráficos
# ============================================

class VisualizationLayer:
    """Crea visualizaciones interactivas de los datos Gold"""

    def __init__(self, gold_dfs: Dict[str, pd.DataFrame]):
        self.gold_dfs = gold_dfs
        self.figures = {}

    def create_power_distribution(self):
        """Gráfico de distribución de niveles de poder"""
        print("📊 VIZ: Creando distribución de poder...")

        summary = self.gold_dfs['pokemon_summary']

        fig = px.histogram(
            summary,
            x='total_stats',
            color='power_level',
            nbins=30,
            title='📈 Distribución de Estadísticas Totales por Nivel de Poder',
            labels={'total_stats': 'Estadísticas Totales', 'count': 'Cantidad de Pokémon'},
            color_discrete_map={
                'Low': '#FF6B6B',
                'Medium': '#FFA726',
                'High': '#66BB6A',
                'Elite': '#AB47BC'
            }
        )

        fig.update_layout(template='plotly_white', height=500)
        self.figures['power_distribution'] = fig
        return fig

    def create_type_analysis_chart(self):
        """Gráfico de análisis por tipo"""
        print("📊 VIZ: Creando análisis por tipo...")

        type_analysis = self.gold_dfs['type_analysis'].head(15)

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=(
                'Cantidad de Pokémon por Tipo',
                'Altura Promedio por Tipo',
                'Peso Promedio por Tipo',
                'Experiencia Promedio por Tipo'
            ),
            specs=[[{"type": "bar"}, {"type": "bar"}],
                   [{"type": "bar"}, {"type": "bar"}]]
        )

        fig.add_trace(
            go.Bar(x=type_analysis['type'], y=type_analysis['pokemon_count'],
                   name='Cantidad', marker_color='#42A5F5'),
            row=1, col=1
        )

        fig.add_trace(
            go.Bar(x=type_analysis['type'], y=type_analysis['avg_height_m'],
                   name='Altura (m)', marker_color='#66BB6A'),
            row=1, col=2
        )

        fig.add_trace(
            go.Bar(x=type_analysis['type'], y=type_analysis['avg_weight_kg'],
                   name='Peso (kg)', marker_color='#FFA726'),
            row=2, col=1
        )

        fig.add_trace(
            go.Bar(x=type_analysis['type'], y=type_analysis['avg_experience'],
                   name='Experiencia', marker_color='#AB47BC'),
            row=2, col=2
        )

        fig.update_layout(
            title_text='🎯 Análisis Completo por Tipo de Pokémon',
            showlegend=False,
            height=800,
            template='plotly_white'
        )

        self.figures['type_analysis'] = fig
        return fig

    def create_stats_heatmap(self):
        """Heatmap de estadísticas"""
        print("📊 VIZ: Creando heatmap de estadísticas...")

        stats_pivot = self.gold_dfs['stats_pivot'].head(20)

        stat_cols = [col for col in stats_pivot.columns
                     if col not in ['pokemon_id', 'pokemon_name', 'Total']]

        heatmap_data = stats_pivot[stat_cols].values
        pokemon_names = stats_pivot['pokemon_name'].values

        fig = go.Figure(data=go.Heatmap(
            z=heatmap_data,
            x=stat_cols,
            y=pokemon_names,
            colorscale='Viridis',
            text=heatmap_data,
            texttemplate='%{text}',
            textfont={"size": 10},
            colorbar=dict(title="Valor")
        ))

        fig.update_layout(
            title='🔥 Heatmap de Estadísticas (Top 20 Pokémon)',
            xaxis_title='Estadística',
            yaxis_title='Pokémon',
            height=700,
            template='plotly_white'
        )

        self.figures['stats_heatmap'] = fig
        return fig

    def create_3d_scatter(self):
        """Scatter 3D: altura vs peso vs experiencia"""
        print("📊 VIZ: Creando scatter 3D...")

        summary = self.gold_dfs['pokemon_summary']

        fig = px.scatter_3d(
            summary,
            x='height',
            y='weight',
            z='base_experience',
            color='types',
            size='total_stats',
            hover_data=['name', 'power_level'],
            title='🌐 Análisis 3D: Altura vs Peso vs Experiencia',
            labels={
                'height': 'Altura (m)',
                'weight': 'Peso (kg)',
                'base_experience': 'Experiencia Base'
            }
        )

        fig.update_layout(height=700, template='plotly_white')
        self.figures['3d_scatter'] = fig
        return fig

    def create_radar_chart(self, pokemon_ids: list = None):
        """Radar chart comparativo de estadísticas"""
        print("📊 VIZ: Creando radar chart...")

        stats_pivot = self.gold_dfs['stats_pivot']

        if pokemon_ids is None:
            top_pokemon = stats_pivot.nlargest(5, 'Total')
        else:
            top_pokemon = stats_pivot[stats_pivot['pokemon_id'].isin(pokemon_ids)]

        stat_cols = [col for col in stats_pivot.columns
                     if col not in ['pokemon_id', 'pokemon_name', 'Total']]

        fig = go.Figure()

        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8']

        for idx, (_, pokemon) in enumerate(top_pokemon.iterrows()):
            fig.add_trace(go.Scatterpolar(
                r=[pokemon[stat] for stat in stat_cols],
                theta=stat_cols,
                fill='toself',
                name=pokemon['pokemon_name'],
                line_color=colors[idx % len(colors)]
            ))

        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 150])),
            showlegend=True,
            title='⭐ Comparación de Estadísticas (Top 5)',
            height=600,
            template='plotly_white'
        )

        self.figures['radar_chart'] = fig
        return fig

    def create_top_pokemon_ranking(self):
        """Ranking de top pokémon"""
        print("📊 VIZ: Creando ranking de pokémon...")

        summary = self.gold_dfs['pokemon_summary'].nlargest(15, 'total_stats')

        fig = px.bar(
            summary,
            y='name',
            x='total_stats',
            color='power_level',
            orientation='h',
            title='🏆 Top 15 Pokémon por Estadísticas Totales',
            labels={'total_stats': 'Estadísticas Totales', 'name': 'Pokémon'},
            text='total_stats',
            color_discrete_map={
                'Low': '#FF6B6B',
                'Medium': '#FFA726',
                'High': '#66BB6A',
                'Elite': '#AB47BC'
            }
        )

        fig.update_traces(texttemplate='%{text}', textposition='outside')
        fig.update_layout(
            height=600,
            template='plotly_white',
            yaxis={'categoryorder': 'total ascending'}
        )

        self.figures['top_ranking'] = fig
        return fig

    def create_size_category_sunburst(self):
        """Sunburst de categorías"""
        print("📊 VIZ: Creando sunburst de categorías...")

        summary = self.gold_dfs['pokemon_summary']

        fig = px.sunburst(
            summary,
            path=['size_category', 'power_level'],
            values='total_stats',
            title='☀️ Distribución por Tamaño y Nivel de Poder',
            color='total_stats',
            color_continuous_scale='RdYlGn'
        )

        fig.update_layout(height=600, template='plotly_white')
        self.figures['sunburst'] = fig
        return fig

    def create_correlation_matrix(self):
        """Matriz de correlación"""
        print("📊 VIZ: Creando matriz de correlación...")

        summary = self.gold_dfs['pokemon_summary']

        numeric_cols = ['height', 'weight', 'base_experience', 'total_stats',
                       'avg_stat', 'max_stat', 'abilities_count']

        corr_matrix = summary[numeric_cols].corr()

        fig = go.Figure(data=go.Heatmap(
            z=corr_matrix.values,
            x=corr_matrix.columns,
            y=corr_matrix.columns,
            colorscale='RdBu',
            zmid=0,
            text=corr_matrix.values.round(2),
            texttemplate='%{text}',
            textfont={"size": 10},
            colorbar=dict(title="Correlación")
        ))

        fig.update_layout(
            title='🔗 Matriz de Correlación de Características',
            height=600,
            template='plotly_white'
        )

        self.figures['correlation'] = fig
        return fig

    def generate_and_show_all(self):
        """Genera y muestra todas las visualizaciones en Colab"""
        print("\n" + "=" * 60)
        print("🎨 GENERANDO Y MOSTRANDO VISUALIZACIONES")
        print("=" * 60 + "\n")

        viz_list = [
            (self.create_top_pokemon_ranking, '🏆 Top 15 Pokémon por Estadísticas'),
            (self.create_power_distribution, '📈 Distribución de Niveles de Poder'),
            (self.create_radar_chart, '⭐ Comparación de Estadísticas (Top 5)'),
            (self.create_type_analysis_chart, '🎯 Análisis Completo por Tipo'),
            (self.create_3d_scatter, '🌐 Análisis 3D: Altura vs Peso vs Experiencia'),
            (self.create_stats_heatmap, '🔥 Heatmap de Estadísticas'),
            (self.create_correlation_matrix, '🔗 Matriz de Correlación'),
            (self.create_size_category_sunburst, '☀️ Distribución por Categorías')
        ]

        for create_func, title in viz_list:
            print(f"\n{'='*60}")
            print(f"{title}")
            print('='*60)
            fig = create_func()
            fig.show()

        print(f"\n✅ {len(self.figures)} visualizaciones mostradas")



In [21]:


# ============================================
# PIPELINE PRINCIPAL
# ============================================

def run_etl_pipeline(pokemon_limit: int = 50, save_files: bool = True, show_visualizations: bool = True):
    """
    Ejecuta el pipeline ETL completo con visualizaciones

    Args:
        pokemon_limit: Cantidad de pokémon a procesar
        save_files: Si True, guarda los archivos CSV/JSON
        show_visualizations: Si True, muestra las visualizaciones en Colab
    """

    print("=" * 60)
    print("🚀 INICIANDO ETL PIPELINE - POKÉMON")
    print("=" * 60)

    start_time = time.time()

    # BRONZE: Extracción
    bronze = BronzeLayer()
    bronze_data = bronze.extract_pokemon_data(limit=pokemon_limit)
    bronze.save_bronze()

    if not bronze_data:
        print("❌ Pipeline abortado: No se pudieron extraer datos")
        return

    # SILVER: Transformación
    silver = SilverLayer(bronze_data)
    silver_dfs = silver.process_all()
    silver.save_silver()

    # GOLD: Agregación
    gold = GoldLayer(silver_dfs)
    gold_dfs = gold.process_all()
    gold.save_gold()

    # VISUALIZATION: Gráficos Interactivos
    viz = VisualizationLayer(gold_dfs)
    viz.generate_and_show_all()

    # Resumen final
    elapsed_time = time.time() - start_time

    print("\n" + "=" * 60)
    print("✨ ETL PIPELINE COMPLETADO")
    print("=" * 60)
    print(f"⏱️  Tiempo de ejecución: {elapsed_time:.2f} segundos")
    print(f"📊 Pokémon procesados: {len(silver_dfs['pokemon_base'])}")
    print(f"📈 Datasets generados:")
    print(f"   - Bronze: 1 archivo JSON")
    print(f"   - Silver: {len(silver_dfs)} archivos CSV")
    print(f"   - Gold: {len(gold_dfs)} archivos CSV")
    print(f"   - Visualizaciones: {len(viz.figures)} gráficos interactivos")

    # Estadísticas generales
    print("\n📊 ESTADÍSTICAS GENERALES:")
    summary = gold_dfs['pokemon_summary']
    print(f"Total de pokémon: {len(summary)}")
    print(f"Altura promedio: {summary['height'].mean():.2f} m")
    print(f"Peso promedio: {summary['weight'].mean():.2f} kg")
    print(f"Experiencia base promedio: {summary['base_experience'].mean():.0f}")
    print(f"\nDistribución por nivel de poder:")
    print(summary['power_level'].value_counts())

    print("=" * 60)

    return {
        'bronze': bronze_data,
        'silver': silver_dfs,
        'gold': gold_dfs,
        'visualizations': viz
    }


# ============================================
# EJECUCIÓN Y VISUALIZACIÓN
# ============================================

if __name__ == "__main__":
   # Ejecutar pipeline completo con visualizaciones
    results = run_etl_pipeline(
        pokemon_limit=50,           # Cantidad de pokémon a procesar
        save_files=True,            # Guardar archivos CSV/JSON
        show_visualizations=True    # Mostrar visualizaciones en Colab
    )

    # Acceder a los resultados si es necesario
    if results:
        print("\n✅ Pipeline ejecutado exitosamente")
        print("📂 Datos disponibles en: results['bronze'], results['silver'], results['gold']")
        print("📊 Visualizaciones disponibles en: results['visualizations']")

🚀 INICIANDO ETL PIPELINE - POKÉMON
🔹 BRONZE: Extrayendo datos de 50 pokémon...
  Extrayendo 50/50: diglett
✅ BRONZE: Extracción completada
💾 BRONZE: Datos guardados en pokemon_bronze.json

🔸 SILVER: Transformando datos base...
✅ SILVER: 50 pokémon procesados
🔸 SILVER: Transformando tipos...
✅ SILVER: 76 relaciones de tipo procesadas
🔸 SILVER: Transformando estadísticas...
✅ SILVER: 300 estadísticas procesadas
🔸 SILVER: Transformando habilidades...
✅ SILVER: 120 habilidades procesadas
💾 SILVER: pokemon_base guardado en pokemon_silver_pokemon_base.csv
💾 SILVER: pokemon_types guardado en pokemon_silver_pokemon_types.csv
💾 SILVER: pokemon_stats guardado en pokemon_silver_pokemon_stats.csv
💾 SILVER: pokemon_abilities guardado en pokemon_silver_pokemon_abilities.csv

🔶 GOLD: Creando resumen de pokémon...
✅ GOLD: Resumen creado con 50 pokémon
🔶 GOLD: Creando análisis por tipo...
✅ GOLD: Análisis de 10 tipos creado
🔶 GOLD: Creando pivot de estadísticas...
✅ GOLD: Pivot creado con 50 pokémon
💾 


📈 Distribución de Niveles de Poder
📊 VIZ: Creando distribución de poder...



⭐ Comparación de Estadísticas (Top 5)
📊 VIZ: Creando radar chart...



🎯 Análisis Completo por Tipo
📊 VIZ: Creando análisis por tipo...



🌐 Análisis 3D: Altura vs Peso vs Experiencia
📊 VIZ: Creando scatter 3D...



🔥 Heatmap de Estadísticas
📊 VIZ: Creando heatmap de estadísticas...



🔗 Matriz de Correlación
📊 VIZ: Creando matriz de correlación...



☀️ Distribución por Categorías
📊 VIZ: Creando sunburst de categorías...


/usr/local/lib/python3.12/dist-packages/plotly/express/_core.py:1727: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

/usr/local/lib/python3.12/dist-packages/plotly/express/_core.py:1727: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




✅ 8 visualizaciones mostradas

✨ ETL PIPELINE COMPLETADO
⏱️  Tiempo de ejecución: 8.97 segundos
📊 Pokémon procesados: 50
📈 Datasets generados:
   - Bronze: 1 archivo JSON
   - Silver: 4 archivos CSV
   - Gold: 3 archivos CSV
   - Visualizaciones: 8 gráficos interactivos

📊 ESTADÍSTICAS GENERALES:
Total de pokémon: 50
Altura promedio: 0.94 m
Peso promedio: 23.41 kg
Experiencia base promedio: 125

Distribución por nivel de poder:
power_level
Low       16
High      15
Medium    13
Elite      6
Name: count, dtype: int64

✅ Pipeline ejecutado exitosamente
📂 Datos disponibles en: results['bronze'], results['silver'], results['gold']
📊 Visualizaciones disponibles en: results['visualizations']
